# LMEnt retention study — explore & load

End-to-end demo for the project *“which properties predict whether a fact is retained vs. forgotten?”*

This notebook loads the **PopQA-KAS** dataset and a **LMEnt-1B-1E** checkpoint, then exercises each feature module on a single fact so you can see the pieces before running the full sweep in `scripts/`.

> Runs on the cluster (needs a GPU + the downloaded checkpoints/corpus). Constants that were inferred from the release are in `config.py`, tagged `# VERIFY`.

In [6]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import config
from src import data as D
print('model  :', config.MODEL_ID)
print('dataset:', config.POPQA_DATASET_ID)
print('checkpoints:', config.CHECKPOINT_STEPS)

ModuleNotFoundError: No module named 'config'

## 1. Load PopQA-KAS and inspect the schema

In [3]:
df = D.load_popqa_df()
print(df.shape)
print(list(df.columns))
df[['id','subj','prop','obj','s_pop','o_pop',
    'subject_num_chunks','answer_num_chunks','num_shared_chunks']].head()

NameError: name 'D' is not defined

In [2]:
df

NameError: name 'df' is not defined

In [4]:
df.iloc[0]

id                                                              4222362
subj                                                      George Rankin
prop                                                         occupation
obj                                                          politician
subj_id                                                         1850297
prop_id                                                              22
obj_id                                                          2834605
s_aliases                                         [George James Rankin]
o_aliases             [political leader, political figure, polit., pol]
s_uri                                                          Q5543720
o_uri                                                            Q82955
s_wiki_title                                              George Rankin
o_wiki_title                                                 Politician
s_pop                                                           

In [ ]:
df.shared_chunks.value_counts()

In [ ]:
# Relation distribution and the combined-frequency the paper fixes.
print(df['prop'].value_counts())
row = df.iloc[0]
print('\nexample fact:', row['subj'], '/', row['prop'], '->', row['obj'])
print('shared_chunks (fact appears):', list(row['shared_chunks'])[:10], '...')

## 2. Load a checkpoint + tokenizer, sanity check

In [ ]:
tokenizer = D.load_tokenizer()
model = D.load_checkpoint(config.FINAL_STEP)   # final checkpoint

prompt = D.qa_prompt(row['question'])
ids = tokenizer(prompt, return_tensors='pt').to(model.device)
out = model.generate(**ids, max_new_tokens=16, do_sample=False)
print(prompt)
print('->', tokenizer.decode(out[0, ids['input_ids'].shape[1]:], skip_special_tokens=True))

## 3. Recall + gold-answer probability for one fact

In [ ]:
from src import recall
answers = recall.acceptable_answers(row)
print('acceptable answers:', answers)
print('correct (substring):', recall.greedy_correct(model, tokenizer, prompt, answers))
cloze = D.cloze_prompt(row['subj'], row['prop']) or prompt
print('cloze prompt       :', cloze)
print('gold_prob          :', recall.gold_answer_logprob(model, tokenizer, cloze, row['obj']))

## 4. Training schedule → timeline features

Needs `batch_indices.npy` (from `LMEnt-Dataset/dataset-cache`). The first call
builds and caches the `chunk_id → steps` inverted index.

In [ ]:
from src import schedule
from src.features import timeline, competition

chunk_to_steps = schedule.build_chunk_to_steps()   # cached after first run
appearances = schedule.steps_for_chunks(row['shared_chunks'], chunk_to_steps)
print('fact appeared at steps:', appearances[:20], '...')

timeline.timeline_features(appearances, checkpoint_step=config.FINAL_STEP)

## 5. Competition / interference features

In [ ]:
competition.competition_features(row, chunk_to_steps,
                                 checkpoint_step=config.FINAL_STEP,
                                 first_learned_step=None)

## 6. Inside the model — logit lens, margin, hidden states, paraphrase

The mechanistic group: resolution depth, answer margin, subject hidden states
(for probing / drift), and paraphrase stability.

In [ ]:
from src.features import internal

print('logit lens:', internal.logit_lens_features(model, tokenizer,
        row['subj'], row['prop'], row['obj']))

hs = internal.subject_hidden_states(model, tokenizer, row['subj'], row['prop'])
print('subject hidden states shape (layers, hidden):',
      None if hs is None else hs.shape)

print('paraphrase stability:', internal.paraphrase_stability(model, tokenizer, row))

## 7. Next steps (run as scripts, not in the notebook)

1. `python scripts/eval_checkpoints.py` — recall + gold_prob for every fact at every checkpoint (one GPU per checkpoint; Slurm array job).
2. `python scripts/build_feature_table.py` — join the sweep to the corpus and emit `panel.parquet` (fact×checkpoint) and `facts.parquet` (per-fact + retention target).
3. Step B EDA: hold combined frequency fixed, compare retained vs. forgotten distributions.
4. Step C: fit an interpretable model on `facts.parquet` (drop the `leaky_*` columns first) and read coefficients / SHAP.

**Representation drift** across checkpoints and the **linear probe** need hidden
states from *multiple* checkpoints; collect `internal.subject_hidden_states` per
checkpoint into `artifacts/internal/` then call `internal.representation_drift`
and `internal.train_layer_probes`.